In [ ]:
import sys
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import decoupler as dc
import matplotlib.pyplot as plt
from gprofiler import GProfiler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
import figstyle as fs

BASE = Path.cwd()
OUT_DIR = BASE / "results" / "02_fibroblast_deg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
adata = ad.read_zarr(BASE / "data" / "integrated_human_annotated.zarr")
fib = adata[(adata.obs["cell_largetype"] == "Fibroblast") & adata.obs["roi_name"].isin(["Cion_Late", "Xray_Late"])].copy()
fib.X = fib.layers["log1p_norm"]
fib.obs["roi_name"] = fib.obs["roi_name"].astype(str)
pd.crosstab(fib.obs["leiden"], fib.obs["roi_name"])

In [ ]:
sc.tl.rank_genes_groups(
    fib,
    "roi_name",
    groups=["Xray_Late"],
    reference="Cion_Late",
    method="wilcoxon",
    pts=True,
)
results_df = sc.get.rank_genes_groups_df(fib, group="Xray_Late").rename(columns={
    "names": "gene", "scores": "stat", "logfoldchanges": "log2FoldChange", "pvals": "pvalue", "pvals_adj": "padj"}).set_index("gene")
pts = fib.uns["rank_genes_groups"]["pts"]
results_df["pct_Xray_Late"] = results_df.index.map(pts["Xray_Late"])
results_df["pct_Cion_Late"] = results_df.index.map(pts["Cion_Late"])
detected = results_df[["pct_Xray_Late", "pct_Cion_Late"]].max(axis=1) >= 0.1

sig = results_df[(results_df["padj"] < 0.05) & (results_df["log2FoldChange"].abs() > 0.5) & detected]
results_df.to_csv(OUT_DIR / "fibroblast_deg_Xray_Late_vs_Cion_Late.csv")
sig.to_csv(OUT_DIR / "fibroblast_deg_Xray_Late_vs_Cion_Late_sig.csv")
print("up in Xray_Late:", (sig["log2FoldChange"] > 0).sum(), "/ up in Cion_Late:", (sig["log2FoldChange"] < 0).sum())

In [ ]:
sig[sig["log2FoldChange"] > 0].sort_values("log2FoldChange", ascending=False).head(20)

In [ ]:
sig[sig["log2FoldChange"] < 0].sort_values("log2FoldChange").head(20)

In [ ]:
LABEL = ["CCN2", "SERPINE1", "LTBP2", "ELN", "FBLN5", "MFAP5", "COL5A1", "COL5A2", "ADAMTS1", "SNAI2", "CDKN1A", "SMOC2"]

def volcano(results_df, xlabel, path, top_genes):
    fig, ax = plt.subplots(figsize=(fs.COL1, 70 * fs.MM), layout="constrained")
    dc.pl.volcano(results_df.dropna(subset=["padj"]), x="log2FoldChange", y="padj",
                  top=top_genes,
                  color_pos=fs.UP, color_neg=fs.DOWN, color_null=fs.LIGHTGREY,
                  kw_scatter={"s": 4, "lw": 0, "rasterized": True}, ax=ax)
    for line in ax.lines:
        line.set(color=fs.GREY, lw=0.4, ls=(0, (2, 2)))
    for text in ax.texts:
        text.set(fontsize=5, fontstyle="italic")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("−log$_{10}$(adjusted $\\it{P}$ value)")
    fs.save(fig, path)
    plt.show()

volcano(results_df[detected], f"log$_2$ fold change ({fs.label('Xray_Late')} / {fs.label('Cion_Late')})", OUT_DIR / "fibroblast_volcano", LABEL)

In [ ]:
up = sig.index[sig["log2FoldChange"] > 0].tolist()
panel = fib.var_names.tolist()
print("up in Xray_Late:", len(up), "/ background (panel):", len(panel))

gp = GProfiler(return_dataframe=True)
go = gp.profile(organism="hsapiens", query=up, sources=["GO:BP", "GO:MF", "GO:CC"],
                significance_threshold_method="fdr", domain_scope="custom", background=panel,
                no_evidences=False)
go.to_csv(OUT_DIR / "fibroblast_go_up_Xray_Late.csv", index=False)
go["source"].value_counts()

In [ ]:
for src in ["GO:BP", "GO:CC", "GO:MF"]:
    top = go[go["source"] == src].head(10).iloc[::-1]
    names = ["\n".join(textwrap.wrap(n, 42)) for n in top["name"]]
    fig, ax = plt.subplots(figsize=(fs.COL1, (len(top) * 5 + 12) * fs.MM), layout="constrained")
    norm = plt.Normalize(top["p_value"].min(), top["p_value"].max())
    ax.barh(range(len(top)), top["intersection_size"], color=plt.cm.viridis_r(norm(top["p_value"])), height=0.7)
    exp = int(np.floor(np.log10(top["p_value"].max())))
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis_r"), ax=ax, shrink=0.6, aspect=15,
                      label=f"Adjusted $\\it{{P}}$ value (×10$^{{{exp}}}$)")
    cb.outline.set_linewidth(0.5)
    cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 10 ** exp:g}"))
    ax.set_yticks(range(len(top)), names)
    ax.tick_params(axis="y", length=0)
    fs.save(fig, OUT_DIR / f"fibroblast_go_up_Xray_Late_{src.split(':')[1]}")
    plt.show()